# OpenFDA Drug Adverse Events ETL Pipeline
## Section 1: Extract
Pull adverse event reports from the OpenFDA REST API

In [2]:
import requests
import json

In [4]:
# OpenFDA base URL for drug adverse events
base_url = "https://api.fda.gov/drug/event.json"

# Pull 10 records to start and inspect the structure
params = {
    "limit": 10
}

response = requests.get(base_url, params=params)
print("Status code:", response.status_code)
print("Keys in response:", response.json().keys())


Status code: 200
Keys in response: dict_keys(['meta', 'results'])


In [5]:
# Look at the meta information
print(json.dumps(response.json()['meta'], indent=2))

{
  "disclaimer": "Do not rely on openFDA to make decisions regarding medical care. While we make every effort to ensure that data is accurate, you should assume all results are unvalidated. We may limit or otherwise restrict your access to the API in line with our Terms of Service.",
  "terms": "https://open.fda.gov/terms/",
  "license": "https://open.fda.gov/license/",
  "last_updated": "2026-04-28",
  "results": {
    "skip": 0,
    "limit": 10,
    "total": 20328575
  }
}


In [6]:
# Look at just the first result to understand the structure
print(json.dumps(response.json()['results'][0], indent=2))

{
  "safetyreportid": "5801206-7",
  "transmissiondateformat": "102",
  "transmissiondate": "20090109",
  "serious": "1",
  "seriousnessdeath": "1",
  "receivedateformat": "102",
  "receivedate": "20080707",
  "receiptdateformat": "102",
  "receiptdate": "20080625",
  "fulfillexpeditecriteria": "1",
  "companynumb": "JACAN16471",
  "primarysource": {
    "reportercountry": "CANADA",
    "qualification": "3"
  },
  "sender": {
    "senderorganization": "FDA-Public Use"
  },
  "receiver": null,
  "patient": {
    "patientonsetage": "26",
    "patientonsetageunit": "801",
    "patientsex": "1",
    "patientdeath": {
      "patientdeathdateformat": null,
      "patientdeathdate": null
    },
    "reaction": [
      {
        "reactionmeddrapt": "DRUG ADMINISTRATION ERROR"
      },
      {
        "reactionmeddrapt": "OVERDOSE"
      }
    ],
    "drug": [
      {
        "drugcharacterization": "1",
        "medicinalproduct": "DURAGESIC-100",
        "drugauthorizationnumb": "019813",
   

In [7]:
def extract_record(report):
    """Extract relevant fields from a single adverse event report"""
    
    # safely pull nested fields
    patient = report.get('patient', {})
    primary_source = report.get('primarysource', {})
    
    # reactions and drugs can be lists so we join them
    reactions = patient.get('reaction', [])
    reaction_list = ', '.join([r.get('reactionmeddrapt', '') for r in reactions])
    
    drugs = patient.get('drug', [])
    drug_list = ', '.join([d.get('medicinalproduct', '') for d in drugs])
    drug_indications = ', '.join([d.get('drugindication', '') for d in drugs])
    
    return {
        'safetyreportid': report.get('safetyreportid'),
        'receivedate': report.get('receivedate'),
        'serious': report.get('serious'),
        'seriousnessdeath': report.get('seriousnessdeath', '0'),
        'country': primary_source.get('reportercountry'),
        'patient_age': patient.get('patientonsetage'),
        'patient_sex': patient.get('patientsex'),
        'reactions': reaction_list,
        'drugs': drug_list,
        'drug_indications': drug_indications
    }

# test it on our first record
first_record = response.json()['results'][0]
extracted = extract_record(first_record)
print(json.dumps(extracted, indent=2))

{
  "safetyreportid": "5801206-7",
  "receivedate": "20080707",
  "serious": "1",
  "seriousnessdeath": "1",
  "country": "CANADA",
  "patient_age": "26",
  "patient_sex": "1",
  "reactions": "DRUG ADMINISTRATION ERROR, OVERDOSE",
  "drugs": "DURAGESIC-100",
  "drug_indications": "DRUG ABUSE"
}


In [8]:
# Apply extract_record to all 10 results
results = response.json()['results']
extracted_records = [extract_record(report) for report in results]

# Check how many we got
print(f"Extracted {len(extracted_records)} records")
print(json.dumps(extracted_records[0], indent=2))

Extracted 10 records
{
  "safetyreportid": "5801206-7",
  "receivedate": "20080707",
  "serious": "1",
  "seriousnessdeath": "1",
  "country": "CANADA",
  "patient_age": "26",
  "patient_sex": "1",
  "reactions": "DRUG ADMINISTRATION ERROR, OVERDOSE",
  "drugs": "DURAGESIC-100",
  "drug_indications": "DRUG ABUSE"
}


In [9]:
import pandas as pd

df = pd.DataFrame(extracted_records)
df.head()

,safetyreportid,receivedate,serious,seriousnessdeath,country,patient_age,patient_sex,reactions,drugs,drug_indications
0,5801206-7,20080707,1,1,CANADA,26,1,"DRUG ADMINISTRATION ERROR, OVERDOSE",DURAGESIC-100,DRUG ABUSE
1,10003300,20140306,1,0,US,77,2,"Vomiting, Diarrhoea, Arthralgia, Headache",BONIVA,OSTEOPOROSIS
2,10003301,20140228,1,0,US,None,2,"Dyspepsia, Renal impairment",IBUPROFEN,PRODUCT USED FOR UNKNOWN INDICATION
3,10003302,20140312,2,0,US,None,1,Drug ineffective,LYRICA,
4,10003304,20140312,2,0,US,None,2,Drug hypersensitivity,"DOXYCYCLINE HYCLATE, TRAMADOL HYDROCHLORIDE, O...",", , , , , , , ,"


In [10]:
def transform_df(df):
    
    # 1. Fix date format
    df['receivedate'] = pd.to_datetime(df['receivedate'], format='%Y%m%d', errors='coerce')
    
    # 2. Map sex codes to readable values
    sex_map = {'1': 'Male', '2': 'Female'}
    df['patient_sex'] = df['patient_sex'].map(sex_map).fillna('Unknown')
    
    # 3. Map serious columns to Yes/No
    df['serious'] = df['serious'].map({'1': 'Yes', '0': 'No'}).fillna('Unknown')
    df['seriousnessdeath'] = df['seriousnessdeath'].map({'1': 'Yes', '0': 'No'}).fillna('No')
    
    # 4. Clean up drug indications - remove trailing commas and extra spaces
    df['drug_indications'] = df['drug_indications'].str.strip(', ').str.strip()
    
    # 5. Convert age to numeric
    df['patient_age'] = pd.to_numeric(df['patient_age'], errors='coerce')
    
    return df

# Apply it
df_clean = transform_df(df.copy())
df_clean.head()

,safetyreportid,receivedate,serious,seriousnessdeath,country,patient_age,patient_sex,reactions,drugs,drug_indications
0,5801206-7,2008-07-07,Yes,Yes,CANADA,26.0,Male,"DRUG ADMINISTRATION ERROR, OVERDOSE",DURAGESIC-100,DRUG ABUSE
1,10003300,2014-03-06,Yes,No,US,77.0,Female,"Vomiting, Diarrhoea, Arthralgia, Headache",BONIVA,OSTEOPOROSIS
2,10003301,2014-02-28,Yes,No,US,NaN,Female,"Dyspepsia, Renal impairment",IBUPROFEN,PRODUCT USED FOR UNKNOWN INDICATION
3,10003302,2014-03-12,Unknown,No,US,NaN,Male,Drug ineffective,LYRICA,
4,10003304,2014-03-12,Unknown,No,US,NaN,Female,Drug hypersensitivity,"DOXYCYCLINE HYCLATE, TRAMADOL HYDROCHLORIDE, O...",


In [11]:
def extract_batch(limit=100, skip=0):
    """Pull a batch of records from OpenFDA API"""
    
    params = {
        "limit": limit,
        "skip": skip
    }
    
    response = requests.get(base_url, params=params)
    
    if response.status_code == 200:
        results = response.json()['results']
        extracted = [extract_record(report) for report in results]
        return pd.DataFrame(extracted)
    else:
        print(f"Error: {response.status_code}")
        return None

In [12]:
df_raw = extract_batch(limit=100, skip=0)
df_clean = transform_df(df_raw.copy())

print(f"Shape: {df_clean.shape}")
df_clean.head()


Shape: (100, 10)


,safetyreportid,receivedate,serious,seriousnessdeath,country,patient_age,patient_sex,reactions,drugs,drug_indications
0,5801206-7,2008-07-07,Yes,Yes,CANADA,26.0,Male,"DRUG ADMINISTRATION ERROR, OVERDOSE",DURAGESIC-100,DRUG ABUSE
1,10003300,2014-03-06,Yes,No,US,77.0,Female,"Vomiting, Diarrhoea, Arthralgia, Headache",BONIVA,OSTEOPOROSIS
2,10003301,2014-02-28,Yes,No,US,NaN,Female,"Dyspepsia, Renal impairment",IBUPROFEN,PRODUCT USED FOR UNKNOWN INDICATION
3,10003302,2014-03-12,Unknown,No,US,NaN,Male,Drug ineffective,LYRICA,
4,10003304,2014-03-12,Unknown,No,US,NaN,Female,Drug hypersensitivity,"DOXYCYCLINE HYCLATE, TRAMADOL HYDROCHLORIDE, O...",


In [13]:
# Quick summary of our clean data
print(f"Total records: {len(df_clean)}")
print(f"\nMissing values per column:")
print(df_clean.isnull().sum())
print(f"\nSerious events: {df_clean['serious'].value_counts().to_dict()}")
print(f"\nDeath cases: {df_clean['seriousnessdeath'].value_counts().to_dict()}")

Total records: 100

Missing values per column:
safetyreportid       0
receivedate          0
serious              0
seriousnessdeath     0
country              0
patient_age         11
patient_sex          0
reactions            0
drugs                0
drug_indications     0
dtype: int64

Serious events: {'Unknown': 66, 'Yes': 34}

Death cases: {'No': 96, 'Yes': 4}


In [15]:
import sys
!{sys.executable} -m pip install boto3

In [17]:
import boto3

# Create an S3 client - boto3 automatically uses the credentials we just configured
s3 = boto3.client('s3')

bucket_name = 'aditi-openfda-etl-pipeline'

# Save our clean dataframe locally first as CSV
df_clean.to_csv('openfda_clean_100.csv', index=False)

# Upload it to S3
s3.upload_file('openfda_clean_100.csv', bucket_name, 'raw/openfda_clean_100.csv')

print("Upload complete")

Upload complete


In [18]:
# Create a folder for Athena query results
s3.put_object(Bucket=bucket_name, Key='athena-results/')
print("Results folder created")

Results folder created


In [19]:
# This is what our Lambda function will look like
# Let's build and test it here in Jupyter first

import requests
import pandas as pd
import boto3
from datetime import datetime

def extract_record(report):
    """Extract relevant fields from a single adverse event report"""
    patient = report.get('patient', {})
    primary_source = report.get('primarysource', {})
    
    reactions = patient.get('reaction', [])
    reaction_list = ', '.join([r.get('reactionmeddrapt', '') for r in reactions])
    
    drugs = patient.get('drug', [])
    drug_list = ', '.join([d.get('medicinalproduct', '') for d in drugs])
    drug_indications = ', '.join([d.get('drugindication', '') for d in drugs])
    
    return {
        'safetyreportid': report.get('safetyreportid'),
        'receivedate': report.get('receivedate'),
        'serious': report.get('serious'),
        'seriousnessdeath': report.get('seriousnessdeath', '0'),
        'country': primary_source.get('reportercountry'),
        'patient_age': patient.get('patientonsetage'),
        'patient_sex': patient.get('patientsex'),
        'reactions': reaction_list,
        'drugs': drug_list,
        'drug_indications': drug_indications
    }

def transform_df(df):
    df['receivedate'] = pd.to_datetime(df['receivedate'], format='%Y%m%d', errors='coerce')
    
    sex_map = {'1': 'Male', '2': 'Female'}
    df['patient_sex'] = df['patient_sex'].map(sex_map).fillna('Unknown')
    
    df['serious'] = df['serious'].map({'1': 'Yes', '0': 'No'}).fillna('Unknown')
    df['seriousnessdeath'] = df['seriousnessdeath'].map({'1': 'Yes', '0': 'No'}).fillna('No')
    
    df['drug_indications'] = df['drug_indications'].str.strip(', ').str.strip()
    df['patient_age'] = pd.to_numeric(df['patient_age'], errors='coerce')
    
    return df

print("Functions loaded successfully")

Functions loaded successfully


In [20]:
def lambda_handler(event, context):
    """
    This is the function Lambda will actually call.
    It extracts, transforms, and loads OpenFDA data to S3.
    """
    
    bucket_name = 'aditi-openfda-etl-pipeline'
    base_url = "https://api.fda.gov/drug/event.json"
    
    # Step 1: Extract
    params = {"limit": 100, "skip": 0}
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        return {
            'statusCode': response.status_code,
            'body': f'Error fetching data from OpenFDA: {response.status_code}'
        }
    
    results = response.json()['results']
    extracted_records = [extract_record(report) for report in results]
    
    # Step 2: Transform
    df = pd.DataFrame(extracted_records)
    df_clean = transform_df(df)
    
    # Step 3: Load
    # Create a unique filename using today's date so each run doesn't overwrite the last
    today = datetime.now().strftime('%Y-%m-%d')
    filename = f'/tmp/openfda_{today}.csv'  # Lambda can only write to /tmp
    s3_key = f'raw/openfda_{today}.csv'
    
    df_clean.to_csv(filename, index=False)
    
    s3 = boto3.client('s3')
    s3.upload_file(filename, bucket_name, s3_key)
    
    return {
        'statusCode': 200,
        'body': f'Successfully loaded {len(df_clean)} records to {s3_key}'
    }

In [21]:
# Test it locally first - simulate what Lambda would do
test_result = lambda_handler(None, None)
print(test_result)

{'statusCode': 200, 'body': 'Successfully loaded 100 records to raw/openfda_2026-06-19.csv'}
